In [ ]:
import math

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import seaborn as sns
import tqdm
from networks import InvariantValueNet, ValueNet
random_key = jax.random.PRNGKey(0)
batch_size = 256

width = 32
value_net_equiv = InvariantValueNet(
    key=jax.random.PRNGKey(0),
    in_size=6 * 7 * 2,
    head_depth=2,
    head_width=width,
    body_depth=2,
    body_width=width,
    body_n_blocks=2,
    embed_dim=width,
    activation=jax.nn.gelu,
    n_actions=7,
    avg_symmetries=False,
    simple=False,
)


value_net_nosym = ValueNet(
    key=jax.random.PRNGKey(0),
    in_size=6 * 7 * 2,
    head_depth=2,
    head_width=width,
    body_depth=2,
    body_width=width,
    body_n_blocks=2,
    embed_dim=width,
    activation=jax.nn.gelu,
    n_actions=7,
    avg_symmetries=False,
    simple=False,
)

value_net_sym = ValueNet(
    key=jax.random.PRNGKey(0),
    in_size=6 * 7 * 2,
    head_depth=2,
    head_width=width,
    body_depth=2,
    body_width=width,
    body_n_blocks=2,
    embed_dim=width,
    activation=jax.nn.gelu,
    n_actions=7,
    avg_symmetries=True,
    simple=False,
)

opt = optax.adam(1e-3)
params_nosym = eqx.filter(value_net_nosym, eqx.is_inexact_array)
params_sym = eqx.filter(value_net_sym, eqx.is_inexact_array)
params_equiv = eqx.filter(value_net_equiv, eqx.is_inexact_array)
opt_state_nosym = opt.init(params_nosym)
opt_state_sym = opt.init(params_sym)
opt_state_equiv = opt.init(params_equiv)

_train_boards = np.load("train_boards.npy")
_train_values = np.load("train_values.npy")
_test_boards = np.load("test_boards.npy")
_test_values = np.load("test_values.npy")

train_boards = jnp.stack([_train_boards == 1, _train_boards == 2], axis=-1).astype(jnp.float32)
test_boards = jnp.stack([_test_boards == 1, _test_boards == 2], axis=-1).astype(jnp.float32)
test_values = jnp.asarray(_test_values, dtype=jnp.float32)
train_values = jnp.asarray(_train_values, dtype=jnp.float32)
@eqx.filter_jit
def get_batches(data, random_key, batch_size, drop_last):
    if not drop_last:
        raise NotImplementedError

    n_samples = jax.tree.leaves(data)[0].shape[0]
    n_batches = math.floor(n_samples / batch_size)

    idx = jax.random.permutation(random_key, jnp.arange(n_samples))[
        : n_batches * batch_size
    ]
    return jax.tree.map(
        lambda x: x[idx].reshape(n_batches, batch_size, *x.shape[1:]), data
    )

@eqx.filter_jit
def do_epoch(random_key, value_net, opt_state, obs, values, opt, batch_size):
    params, static = eqx.partition(value_net, eqx.is_inexact_array)

    data = get_batches((obs, values), random_key, batch_size, drop_last=True)

    def body_fn(carry, x):
        params, opt_state = carry
        obs, values = x
        value_net = eqx.combine(params, static)

        (_, metrics), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(
            value_net, obs, values
        )
        updates, opt_state = opt.update(grads, opt_state)
        value_net = eqx.apply_updates(value_net, updates)
        params = eqx.filter(value_net, eqx.is_inexact_array)
        return (params, opt_state), (metrics)

    (params, opt_state), metrics = jax.lax.scan(body_fn, (params, opt_state), data)
    value_net = eqx.combine(params, static)
    return value_net, opt_state, metrics


@eqx.filter_jit
def loss_fn(value_net, obs, values):
    preds = jax.vmap(value_net.forward)(obs)
    loss = jnp.mean((preds - values) ** 2)
    return loss, {"value_loss":loss}
rows = []
n_epochs = 1
k = 0

eval_loss_nosym, _ = loss_fn(value_net_nosym, test_boards, test_values)
eval_loss_sym, _ = loss_fn(value_net_sym, test_boards, test_values)

value_net_nosym_evalsym = eqx.combine(
    eqx.filter(value_net_nosym, eqx.is_inexact_array),
    eqx.filter(value_net_sym, lambda x: not eqx.is_inexact_array(x)),
)
eval_loss_nosym_evalsym, _ = loss_fn(value_net_nosym_evalsym, test_boards, test_values)
eval_loss_equiv, _ = loss_fn(value_net_equiv, test_boards, test_values)
row = {"value": float(eval_loss_nosym), "step": k, "metric": "eval_nosym"}
rows.append(row)
row = {"value": float(eval_loss_sym), "step": k, "metric": "eval_sym"}
rows.append(row)
row = {
    "value": float(eval_loss_nosym_evalsym),
    "step": k,
    "metric": "eval_nosym_inference_sym",
}
rows.append(row)
row = {
    "value": float(eval_loss_equiv),
    "step": k,
    "metric": "eval_equiv",
}
rows.append(row)

for epoch in tqdm.trange(n_epochs):
    random_key, subkey = jax.random.split(random_key)
    value_net_nosym, opt_state_nosym, metrics_nosym = do_epoch(
        subkey,
        value_net_nosym,
        opt_state_nosym,
        train_boards,
        train_values,
        opt,
        batch_size,
    )

    value_net_sym, opt_state_sym, metrics_sym = do_epoch(
        subkey,
        value_net_sym,
        opt_state_sym,
        train_boards,
        train_values,
        opt,
        batch_size,
    )
    value_net_equiv, opt_state_equiv, metrics_equiv = do_epoch(
        subkey,
        value_net_equiv,
        opt_state_equiv,
        train_boards,
        train_values,
        opt,
        batch_size,
    )

    n_updates = len(metrics_sym["value_loss"])

    add_rows = [
        {"value": float(m), "step": k + i, "metric": "train_sym"}
        for (i, m) in enumerate(metrics_sym["value_loss"])
    ]
    rows += add_rows

    add_rows = [
        {"value": float(m), "step": k + i, "metric": "train_nosym"}
        for (i, m) in enumerate(metrics_nosym["value_loss"])
    ]
    rows += add_rows

    add_rows = [
        {"value": float(m), "step": k + i, "metric": "train_equiv"}
        for (i, m) in enumerate(metrics_equiv["value_loss"])
    ]
    rows += add_rows

    eval_loss_nosym, _ = loss_fn(value_net_nosym, test_boards, test_values)
    eval_loss_sym, _ = loss_fn(value_net_sym, test_boards, test_values)
    eval_loss_equiv, _ = loss_fn(value_net_equiv, test_boards, test_values)

    value_net_nosym_evalsym = eqx.combine(
        eqx.filter(value_net_nosym, eqx.is_inexact_array),
        eqx.filter(value_net_sym, lambda x: not eqx.is_inexact_array(x)),
    )
    eval_loss_nosym_evalsym, _ = loss_fn(
        value_net_nosym_evalsym, test_boards, test_values
    )
    k += n_updates
    row = {"value": float(eval_loss_nosym), "step": k, "metric": "eval_nosym"}
    rows.append(row)
    row = {"value": float(eval_loss_sym), "step": k, "metric": "eval_sym"}
    rows.append(row)
    row = {
        "value": float(eval_loss_nosym_evalsym),
        "step": k,
        "metric": "eval_nosym_inference_sym",
    }
    rows.append(row)
    row = {
        "value": float(eval_loss_equiv),
        "step": k,
        "metric": "eval_equiv",
    }
    rows.append(row)

df = pd.DataFrame(rows)
sns.set_theme()
plot_df = df[df.metric.str.contains("train")]

sns.lineplot(data=plot_df, x="step", y="value", hue="metric")

sns.set_theme()
plot_df = df[df.metric.str.contains("eval")]
plot_df.loc[:, "metric"] = plot_df["metric"].apply(lambda x: x.replace("eval_", ""))
sns.lineplot(data=plot_df, x="step", y="value", hue="metric")
plt.ylabel("eval loss")
plt.xlabel("gradient steps")